In [19]:
import json
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import os

import dspy

In [2]:
from loguru import logger

logger.remove()
logger.add(lambda _: None, level="CRITICAL")

1

In [3]:
lm = dspy.LM(
    "hosted_vllm/ibm-granite/granite-4.1-8b",
    api_base="http://boris:30000/v1",
    cache=False,
)
dspy.configure(lm=lm)

In [4]:
def load_jsonl(
    path: str | Path,
    *,
    input_keys: tuple[str, ...] = ("question", "reference_datetime"),
) -> list[dspy.Example]:
    examples = []

    with Path(path).open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                row = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON in {path}, line {line_number}"
                ) from error

            example = dspy.Example(**row).with_inputs(*input_keys)
            examples.append(example)

    return examples

# EDA

In [5]:
trainset = load_jsonl("../data/train.jsonl")
valset = load_jsonl("../data/validation.jsonl")

In [6]:
print(trainset[0])

Example({'id': 'train-001', 'family_id': 'train-now-001', 'scenario': 'schedule_now_alias', 'question': 'Next Caltrain from Sunnyvale to SF?', 'reference_datetime': '2026-08-24T15:00:00-07:00', 'is_schedule_question': True, 'departure_station': 'sunnyvale', 'arrival_station': 'san francisco', 'departure_time': '2026-08-24T15:00:00-07:00', 'time_tolerance_minutes': 5}) (input_keys={'reference_datetime', 'question'})


In [7]:
print(len(trainset))
print(len(valset))

88
91


In [8]:
from caltrain_bot.question_analysis import (
    CaltrainScheduleHelper,
    build_station_extraction_signature,
    datetime_calculator,
    QuestionsClassifier,
    UnsupportedQuestion,
    ScheduleQuestion,
)

In [9]:
stations = (
    "22nd street",
    "bayshore",
    "belmont",
    "blossom hill",
    "broadway",
    "burlingame",
    "california avenue",
    "capitol",
    "college park",
    "gilroy",
    "hayward park",
    "hillsdale",
    "lawrence",
    "menlo park",
    "millbrae",
    "morgan hill",
    "mountain view",
    "palo alto",
    "redwood city",
    "san antonio",
    "san bruno",
    "san carlos",
    "san francisco",
    "san jose diridon",
    "san martin",
    "san mateo",
    "santa clara",
    "south san francisco",
    "sunnyvale",
    "tamien",
)

In [10]:
prog = CaltrainScheduleHelper(
    question_classifier=dspy.Predict(QuestionsClassifier),
    stations_departure_time_extractor=dspy.ReAct(
        signature=build_station_extraction_signature(stations),
        tools=[datetime_calculator],
        max_iters=10,
    ),
    stations=stations,
)

In [11]:
prog(
    question="caltrain from sf to palo alto",
    reference_datetime=datetime.now(ZoneInfo("America/Los_Angeles")).isoformat(),
)

ScheduleQuestion(departure_station='san francisco', arrival_station='palo alto', departure_time=datetime.datetime(2026, 8, 25, 10, 6, 23, 673437, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))))

In [20]:
from dspy.teleprompt import GEPA
from dspy.teleprompt.gepa.gepa_utils import DSPyTrace

# Baseline

In [23]:
FIELD_REWARD = 1.0 / 3.0


def parse_datetime(value: object) -> datetime | None:
    if isinstance(value, datetime):
        parsed = value
    elif isinstance(value, str):
        try:
            parsed = datetime.fromisoformat(
                value.removesuffix("Z") + ("+00:00" if value.endswith("Z") else "")
            )
        except ValueError:
            return None
    else:
        return None

    return parsed if parsed.utcoffset() is not None else None


def stations_match(expected: object, predicted: object) -> bool:
    def normalize(value: object) -> str | None:
        if not isinstance(value, str):
            return None
        return " ".join(value.casefold().split()) or None

    expected = normalize(expected)
    predicted = normalize(predicted)

    return expected is not None and predicted == expected


def times_match(
    expected: object,
    predicted: object,
    tolerance_minutes: float,
) -> bool:
    expected_time = parse_datetime(expected)
    predicted_time = parse_datetime(predicted)

    if expected_time is None or predicted_time is None:
        return False

    tolerance = timedelta(minutes=tolerance_minutes)
    return abs(predicted_time - expected_time) <= tolerance


def baseline_metric(
    gold: dspy.Example,
    pred: ScheduleQuestion | UnsupportedQuestion,
    trace: DSPyTrace | None = None,
    pred_name: str | None = None,
    pred_trace: DSPyTrace | None = None,
) -> float:
    if not gold.is_schedule_question:
        return float(isinstance(pred, UnsupportedQuestion))

    if not isinstance(pred, ScheduleQuestion):
        return 0.0

    field_results = (
        stations_match(gold.departure_station, pred.departure_station),
        stations_match(gold.arrival_station, pred.arrival_station),
        times_match(
            gold.departure_time,
            pred.departure_time,
            gold.time_tolerance_minutes,
        ),
    )

    if all(field_results):
        return 1.0

    reward = 0.0
    for is_correct in field_results:
        if is_correct:
            reward += FIELD_REWARD

    return reward

In [13]:
evaluator = dspy.Evaluate(
    devset=trainset + valset,
    num_threads=2,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
    max_errors=5,
    failure_score=0.0,
)

In [14]:
merged_set_result = evaluator(prog, metric=baseline_metric)

Average Metric: 112.00 / 179 (62.6%): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 179/179 [14:49<00:00,  4.97s/it]

2026/08/25 10:21:18 INFO dspy.evaluate.evaluate: Average Metric: 112.00000000000003 / 179 (62.6%)


,id,family_id,scenario,question,reference_datetime,is_schedule_question,departure_station,arrival_station,departure_time,time_tolerance_minutes,prediction,baseline_metric
0,train-001,train-now-001,schedule_now_alias,Next Caltrain from Sunnyvale to SF?,2026-08-24T15:00:00-07:00,True,sunnyvale,san francisco,2026-08-24T15:00:00-07:00,5.0,"ScheduleQuestion(departure_station='sunnyvale', arrival_station='s...",✔️ [0.667]
1,train-002,train-now-002,schedule_now_reversed_alias,I need to get to 4th and King from Palo Alto. What's the next one?,2026-08-24T15:00:00-07:00,True,palo alto,san francisco,2026-08-24T15:00:00-07:00,5.0,"ScheduleQuestion(departure_station='palo alto', arrival_station='m...",✔️ [0.667]
2,train-003,train-now-003,schedule_now_alias,Catching the next train at Diridon going to Mountain View.,2026-08-24T15:00:00-07:00,True,san jose diridon,mountain view,2026-08-24T15:00:00-07:00,5.0,UnsupportedQuestion(),✔️ [0.000]
3,train-004,train-now-004,schedule_now,Any train leaving Redwood City for Millbrae right now?,2026-08-24T15:00:00-07:00,True,redwood city,millbrae,2026-08-24T15:00:00-07:00,5.0,"ScheduleQuestion(departure_station='redwood city', arrival_station...",✔️ [0.333]
4,train-005,train-now-005,schedule_now_alias,"ASAP, 22nd St to San Mateo - what can I catch?",2026-08-24T15:00:00-07:00,True,22nd street,san mateo,2026-08-24T15:00:00-07:00,5.0,"ScheduleQuestion(departure_station='22nd street', arrival_station=...",✔️ [0.667]


In [15]:
merged_set_result.score

62.57

# Optimizer

In [24]:
prompt_lm = dspy.LM(
    model="openrouter/moonshotai/kimi-k3",
    api_base="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [27]:
prompt_lm("Hello!")

[{'text': 'Hello! How can I help you today?',
  'reasoning_content': 'The user has simply said "Hello!" - a friendly greeting. This is about as simple as messages get. There\'s no complex question, no task, no problem to solve. Just a greeting.\n\nWhat\'s the appropriate response here? A warm, friendly greeting back. I shouldn\'t overload them with information or ask a bunch of questions. Just a natural, warm hello and maybe an invitation to share what they\'d like help with.\n\nFormat considerations: This should be very short. A greeting deserves a greeting back - maybe one or two sentences. No headers, no bullet points, no lengthy explanation of what I can do. Just a natural conversational response. Something like "Hello! How can I help you today?" or a variation that feels warm and natural.\n\nI could be slightly more personable than the robotic "How may I assist you?" - something friendly and open.'}]

In [ ]:
gepa_optimizer = GEPA(metric=baseline_metric, reflection_lm=prompt_lm, auto="medium")
gepa_prog = gepa_optimizer.compile(student=prog, trainset=trainset, valset=valset)

In [35]:
gepa_prog.save("../data/prog_caltrain_schedule_helper_gepa_medium", save_program=True)

2026/08/25 15:51:58 WARNING dspy.primitives.base_module: Loading untrusted .pkl files can run arbitrary code, which may be dangerous. To avoid this, prefer saving using json format using module.save("module.json").


In [52]:
gepa_prog._question_classifier

Predict(StringSignature(question -> is_schedule_question
    instructions='Classify whether a question contains a self-contained Caltrain schedule lookup.\n\nA supported question identifies both departure and arrival stations. A departure\ntime is optional. Reject questions that omit either station or depend on prior\nconversational context.'
    question = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Question:', 'desc': '${question}'})
    is_schedule_question = Field(annotation=bool required=True json_schema_extra={'desc': 'True only when both departure and arrival stations can be resolved from this question; the departure time may be omitted', '__dspy_field_type': 'output', 'prefix': 'Is Schedule Question:'})
))

In [53]:
gepa_prog._stations_departure_time_extractor

react = Predict(StringSignature(question, reference_datetime, trajectory -> next_thought, next_tool_name, next_tool_args
    instructions='Extract a Caltrain route and an optional departure time. Return null for a station that cannot be resolved from the question. Return null for the departure time when the question does not specify one.\n\nYou are an Agent. In each episode, you will be given the fields `question`, `reference_datetime` as input. And you can see your past trajectory so far.\nYour goal is to use one or more of the supplied tools to collect any necessary information for producing `departure_station`, `arrival_station`, `departure_time`.\n\nTo do this, you will interleave next_thought, next_tool_name, and next_tool_args in each turn, and also when finishing the task.\nAfter each tool call, you receive a resulting observation, which gets appended to your trajectory.\n\nWhen writing next_thought, you may reason about the current situation and plan for future steps.\nWhen sel

# Re-Evaluate Merged Set

In [54]:
lm = dspy.LM(
    model="openrouter/ibm-granite/granite-4.1-8b",
    api_base="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
dspy.configure(lm=lm)

In [55]:
evaluator = dspy.Evaluate(
    devset=trainset + valset,
    num_threads=10,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
    max_errors=5,
    failure_score=0.0,
)

In [56]:
prog_gepa_optimized_merged_set_result = evaluator(gepa_prog, metric=baseline_metric)

Average Metric: 143.67 / 179 (80.3%): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 179/179 [01:40<00:00,  1.78it/s]

2026/08/25 16:30:19 INFO dspy.evaluate.evaluate: Average Metric: 143.6666666666667 / 179 (80.3%)


,id,family_id,scenario,question,reference_datetime,is_schedule_question,departure_station,arrival_station,departure_time,time_tolerance_minutes,prediction,baseline_metric
0,train-001,train-now-001,schedule_now_alias,Next Caltrain from Sunnyvale to SF?,2026-08-24T15:00:00-07:00,True,sunnyvale,san francisco,2026-08-24T15:00:00-07:00,5.0,"ScheduleQuestion(departure_station='sunnyvale', arrival_station='s...",✔️ [0.667]
1,train-002,train-now-002,schedule_now_reversed_alias,I need to get to 4th and King from Palo Alto. What's the next one?,2026-08-24T15:00:00-07:00,True,palo alto,san francisco,2026-08-24T15:00:00-07:00,5.0,"ScheduleQuestion(departure_station='palo alto', arrival_station='s...",✔️ [1.000]
2,train-003,train-now-003,schedule_now_alias,Catching the next train at Diridon going to Mountain View.,2026-08-24T15:00:00-07:00,True,san jose diridon,mountain view,2026-08-24T15:00:00-07:00,5.0,"ScheduleQuestion(departure_station='san jose diridon', arrival_sta...",✔️ [1.000]
3,train-004,train-now-004,schedule_now,Any train leaving Redwood City for Millbrae right now?,2026-08-24T15:00:00-07:00,True,redwood city,millbrae,2026-08-24T15:00:00-07:00,5.0,"ScheduleQuestion(departure_station='redwood city', arrival_station...",✔️ [1.000]
4,train-005,train-now-005,schedule_now_alias,"ASAP, 22nd St to San Mateo - what can I catch?",2026-08-24T15:00:00-07:00,True,22nd street,san mateo,2026-08-24T15:00:00-07:00,5.0,UnsupportedQuestion(),✔️ [0.000]


In [57]:
prog_gepa_optimized_merged_set_result.score

80.26